# 🔗 LangChain: Zero to Hero — A Guided Lab

LangChain is the most popular framework for composing LLM applications: prompt templates,
**chains** (piping steps together), memory, retrieval, and tool-using **agents**. This lab
teaches the *core mental models* by building tiny working versions of each concept, so you
understand what LangChain does under the hood — then shows the real LangChain syntax alongside.

**Runs 100% offline.** We build minimal stand-ins (`Runnable`, `PromptTemplate`, mock LLM) with
the *same shapes* as real LangChain, so the concepts transfer directly.

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ ASCII diagram → 🔬 Example →
⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. What LangChain is (and the "Runnable" idea)
2. Prompt templates
3. The LLM component
4. Chains & LCEL (the pipe `|` operator)
5. Output parsers (structured results)
6. Memory (stateful conversations)
7. Retrieval chains (RAG in LangChain)
8. Tools
9. Agents (LLM decides which tool to call)
10. 🏆 Capstone: a tool-using assistant chain


In [ ]:
import re, json
from collections import Counter

# ---- Minimal LangChain-shaped primitives (stand-ins for the real library) ----
class Runnable:
    """Everything in LangChain is a Runnable: it has .invoke(input).
    The pipe operator | chains them: (a | b).invoke(x) == b.invoke(a.invoke(x))."""
    def invoke(self, x): raise NotImplementedError
    def __or__(self, other): return _Piped(self, other)

class _Piped(Runnable):
    def __init__(self, first, second): self.first, self.second = first, second
    def invoke(self, x): return self.second.invoke(self.first.invoke(x))

class RunnableLambda(Runnable):
    def __init__(self, fn): self.fn = fn
    def invoke(self, x): return self.fn(x)

print("LangChain-shaped primitives ready (Runnable, pipe, RunnableLambda).")

---
## Chapter 1 — What LangChain Is (the Runnable idea)

📖 **Theory.** LangChain's big idea: wrap every step (prompt, LLM, parser, tool, retriever) in a
common interface called a **Runnable** — anything with an `.invoke(input)` method. Because they
share an interface, you can **pipe** them together into a **chain** with the `|` operator, like
Unix pipes.

🖼️ **Diagram — everything is a Runnable in a pipe**
```
 input ─►[ prompt ]─►[ llm ]─►[ parser ]─► output
            │            │          │
         Runnable    Runnable   Runnable   (all share .invoke)
         chained with |  |  |
```

🧠 **Mental model.** A LangChain app is a **pipeline of Runnables**. Master "everything is a
Runnable you can pipe" and the whole framework clicks.


In [ ]:
# a trivial chain: upper-case then exclaim -- shows the pipe pattern
upper = RunnableLambda(lambda s: s.upper())
excited = RunnableLambda(lambda s: s + "!!!")
chain = upper | excited
print(chain.invoke("hello langchain"))

⚡ **Pro tip.** In real LangChain this is called **LCEL** (LangChain Expression Language). The
`|` you just used is the exact same operator the real library uses.

### ✏️ Your Turn 1.1
Build a chain of three `RunnableLambda`s that: (1) strips whitespace, (2) lowercases, (3)
replaces spaces with hyphens — a tiny "slugify" chain. Test on `"  Hello World  "`.

In [ ]:
slugify = None
print(slugify.invoke("  Hello World  ") if slugify else None)

✅ **Solution**
```python
slugify = (RunnableLambda(str.strip)
           | RunnableLambda(str.lower)
           | RunnableLambda(lambda s: s.replace(" ", "-")))
# -> "hello-world"
```

---
## Chapter 2 — Prompt Templates

📖 **Theory.** Hardcoding prompts is brittle. A **PromptTemplate** is a reusable string with
`{variables}` filled in at runtime. It keeps prompt engineering separate from application logic.

🖼️ **Diagram — template + variables → prompt**
```
 "Translate '{text}' into {language}."  +  {text:"hi", language:"French"}
                     │
                     ▼
      "Translate 'hi' into French."
```


In [ ]:
class PromptTemplate(Runnable):
    def __init__(self, template): self.template = template
    def invoke(self, variables): return self.template.format(**variables)

translate_prompt = PromptTemplate("Translate '{text}' into {language}. Reply with only the translation.")
print(translate_prompt.invoke({"text": "good morning", "language": "Spanish"}))

⚠️ **Common trap.** A missing variable raises `KeyError` at runtime. Keep your template
variables and the dict you pass in synchronized (real LangChain validates `input_variables`).

### ✏️ Your Turn 2.1
Create a `PromptTemplate` for summarization: it takes `{text}` and `{max_words}` and asks for a
summary in at most that many words. Invoke it with sample values.

In [ ]:
summary_prompt = None
print(summary_prompt.invoke({"text": "long article...", "max_words": 20}) if summary_prompt else None)

✅ **Solution**
```python
summary_prompt = PromptTemplate("Summarize the following in at most {max_words} words:\n{text}")
```

---
## Chapter 3 — The LLM Component

📖 **Theory.** The LLM is just another **Runnable**: text in, text out. Wrapping it in the
common interface is what lets it slot into a chain between a prompt and a parser.


In [ ]:
class MockChatLLM(Runnable):
    """Stand-in for ChatOpenAI / ChatGoogleGenerativeAI. invoke(prompt_str) -> str."""
    def invoke(self, prompt):
        p = prompt.lower()
        if "translate" in p and "spanish" in p:
            return "buenos días"
        if "summarize" in p:
            return "A concise summary of the input text."
        if "json" in p or "category" in p:
            return json.dumps({"category": "billing", "priority": "high"})
        if "capital of france" in p:
            return "Paris"
        return "This is a mock LLM response."

llm = MockChatLLM()

# a real chain: prompt | llm
translate_chain = translate_prompt | llm
print(translate_chain.invoke({"text": "good morning", "language": "Spanish"}))

🧠 **Mental model.** `prompt | llm` reads left-to-right: fill the template, then feed the
resulting string to the model. This two-step chain is the backbone of almost every LangChain app.

### ✏️ Your Turn 3.1
Pipe your `summary_prompt` (Ch.2) into `llm` and invoke the resulting chain on some text.

In [ ]:
# summary_chain = summary_prompt | llm ; invoke it


✅ **Solution**
```python
summary_chain = summary_prompt | llm
print(summary_chain.invoke({"text": "Some long text here.", "max_words": 15}))
```

---
## Chapter 4 — Chains & LCEL (the pipe operator)

📖 **Theory.** **LCEL** composes Runnables with `|` into readable, reusable pipelines. A classic
chain is `prompt | llm | parser`. Chains can also branch and merge, but the linear pipe covers
most needs.

🖼️ **Diagram — the classic three-stage chain**
```
 {vars} ─►[ PromptTemplate ]─► prompt str ─►[ LLM ]─► raw text ─►[ Parser ]─► clean result
```


In [ ]:
# Add a parser stage that strips and title-cases the model output
clean = RunnableLambda(lambda s: s.strip().capitalize())
full_chain = translate_prompt | llm | clean
print(full_chain.invoke({"text": "good morning", "language": "Spanish"}))

⚡ **Pro tip.** Keep each stage doing **one** thing (single responsibility). Small Runnables
are easy to test in isolation and recombine — the whole point of LCEL.

### ✏️ Your Turn 4.1
Build a 3-stage chain: a prompt asking for the capital of `{country}` → `llm` → a parser that
wraps the answer as `"The capital is: X"`. Invoke with `{"country": "France"}`.

In [ ]:
capital_chain = None
print(capital_chain.invoke({"country": "France"}) if capital_chain else None)

✅ **Solution**
```python
capital_prompt = PromptTemplate("What is the capital of {country}?")
wrap = RunnableLambda(lambda s: f"The capital is: {s.strip()}")
capital_chain = capital_prompt | llm | wrap
```

---
## Chapter 5 — Output Parsers

📖 **Theory.** LLMs return text, but apps need structured data. An **output parser** is a
Runnable that converts raw text into a typed object (dict, list, etc.) — with error handling for
malformed output.

🖼️ **Diagram — parser closes the loop to real data**
```
 LLM text  '{"category":"billing"}'  ─►[ JsonOutputParser ]─► {"category": "billing"}  (dict)
```


In [ ]:
class JsonOutputParser(Runnable):
    def invoke(self, text):
        text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"error": "could not parse", "raw": text}

classify_prompt = PromptTemplate(
    'Classify this ticket. Respond ONLY with JSON {{"category":..,"priority":..}}.\nTicket: {ticket}')
classify_chain = classify_prompt | llm | JsonOutputParser()
result = classify_chain.invoke({"ticket": "I was double charged"})
print(result, "| type:", type(result).__name__)
print("category field:", result["category"])

⚠️ **Common trap.** Note the **doubled braces** `{{...}}` in the template above — with
Python's `.format`, literal braces must be escaped or they're read as variables. Real LangChain
templates have the same rule.

### ✏️ Your Turn 5.1
Add a `RunnableLambda` stage after the parser that returns just the `"priority"` value from the
parsed dict. Build `classify_prompt | llm | JsonOutputParser() | that_lambda`.

In [ ]:
priority_only = None
print(priority_only.invoke({"ticket": "urgent billing issue"}) if priority_only else None)

✅ **Solution**
```python
get_priority = RunnableLambda(lambda d: d.get("priority", "unknown"))
priority_only = classify_prompt | llm | JsonOutputParser() | get_priority
```

---
## Chapter 6 — Memory (stateful conversations)

📖 **Theory.** Chains are stateless by default. **Memory** stores conversation history and
injects it into each new prompt, so the model has context from prior turns.

🖼️ **Diagram — memory wraps the chain**
```
 history + new input ─►[ prompt(history, input) ]─►[ llm ]─► reply
      ▲                                                        │
      └──────────────── append (input, reply) ◄───────────────┘
```


In [ ]:
class ConversationMemory:
    def __init__(self): self.history = []
    def load(self): return "\n".join(f"{r}: {c}" for r, c in self.history)
    def save(self, user, ai): self.history += [("Human", user), ("AI", ai)]

class ConversationChain:
    def __init__(self, llm, memory): self.llm, self.memory = llm, memory
    def invoke(self, user_input):
        prompt = f"Conversation so far:\n{self.memory.load()}\n\nHuman: {user_input}\nAI:"
        reply = self.llm.invoke(prompt)
        self.memory.save(user_input, reply)
        return reply

convo = ConversationChain(llm, ConversationMemory())
convo.invoke("What is the capital of France?")
convo.invoke("Tell me more")
print("stored turns:", len(convo.memory.history))
print(convo.memory.load()[:120], "...")

⚡ **Pro tip.** Unbounded memory eventually overflows the context window. Real apps use
**windowed** memory (last k turns) or **summary** memory (compress old turns) — exactly the
trade-off you saw in the LLM APIs lab.

### ✏️ Your Turn 6.1
Add a `windowed_load(k)` method to `ConversationMemory` that returns only the last `k` turns
(2k lines). Test it after several exchanges.

In [ ]:
# add windowed_load(k) that returns only the last k (Human,AI) pairs


✅ **Solution**
```python
def windowed_load(self, k=2):
    recent = self.history[-2*k:]
    return "\n".join(f"{r}: {c}" for r, c in recent)
ConversationMemory.windowed_load = windowed_load
```

---
## Chapter 7 — Retrieval Chains (RAG in LangChain)

📖 **Theory.** LangChain makes RAG a chain: a **retriever** Runnable fetches context, which is
formatted into the prompt, then the LLM answers. It's the RAG pipeline from the previous lab,
expressed as piped Runnables.

🖼️ **Diagram — retrieval chain**
```
 question ─►[ retriever ]─► chunks ─►[ format context+question ]─►[ llm ]─► grounded answer
```


In [ ]:
# a tiny keyword retriever as a Runnable
knowledge = [
    "Returns are accepted within 30 days with a receipt.",
    "Refunds take 5 to 7 business days.",
    "Cancel subscriptions in Account Settings.",
]
class Retriever(Runnable):
    def __init__(self, docs, k=2): self.docs, self.k = docs, k
    def invoke(self, query):
        qt = set(re.findall(r"[a-z]+", query.lower()))
        scored = [(d, len(qt & set(re.findall(r"[a-z]+", d.lower())))) for d in self.docs]
        return [d for d, s in sorted(scored, key=lambda x: -x[1])[:self.k] if s > 0]

retriever = Retriever(knowledge)

def rag_invoke(question):
    chunks = retriever.invoke(question)
    context = "\n".join(f"- {c}" for c in chunks) or "No relevant context."
    prompt = f"Answer only from context.\nContext:\n{context}\n\nQuestion: {question}"
    return llm.invoke(prompt), chunks

ans, used = rag_invoke("how long do refunds take?")
print("retrieved:", used)

### ✏️ Your Turn 7.1
Turn `rag_invoke` into a proper piped chain: build a `RunnableLambda` that takes a question and
returns the formatted prompt string, then pipe it into `llm`.

In [ ]:
# build: format_step = RunnableLambda(...) ; rag_chain = format_step | llm


✅ **Solution**
```python
def format_step_fn(question):
    chunks = retriever.invoke(question)
    context = "\n".join(f"- {c}" for c in chunks) or "No relevant context."
    return f"Answer only from context.\nContext:\n{context}\n\nQuestion: {question}"
rag_chain = RunnableLambda(format_step_fn) | llm
print(rag_chain.invoke("how long do refunds take?"))
```

---
## Chapter 8 — Tools

📖 **Theory.** A **tool** is a function the LLM can call to do something it can't do alone:
math, web lookup, database queries, sending email. Each tool has a **name**, a **description**
(so the LLM knows when to use it), and a **function**.

🖼️ **Diagram — a tool**
```
 Tool(name="calculator",
      description="evaluate a math expression",
      func=lambda expr: eval(expr))
```


In [ ]:
class Tool:
    def __init__(self, name, description, func):
        self.name, self.description, self.func = name, description, func
    def run(self, arg): return self.func(arg)

def safe_calc(expr):
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expr): return "invalid expression"
    try: return str(eval(expr))
    except Exception: return "error"

tools = [
    Tool("calculator", "evaluate a math expression like '2 + 3 * 4'", safe_calc),
    Tool("word_count", "count the words in a text", lambda t: str(len(t.split()))),
    Tool("uppercase", "convert text to uppercase", lambda t: t.upper()),
]
for t in tools:
    print(f"{t.name}: {t.description}")
print("\ncalculator(2+3*4) =>", tools[0].run("2 + 3 * 4"))

⚠️ **Common trap.** Never `eval` untrusted input directly — we whitelist characters in
`safe_calc`. Real tools must validate/sandbox their inputs; an LLM *will* eventually pass
something unexpected.

### ✏️ Your Turn 8.1
Add a `reverse` tool that reverses a string, then call it via `.run("hello")`.

In [ ]:
reverse_tool = None
print(reverse_tool.run("hello") if reverse_tool else None)

✅ **Solution**
```python
reverse_tool = Tool("reverse", "reverse a string", lambda s: s[::-1])
```

---
## Chapter 9 — Agents (LLM chooses the tool)

📖 **Theory.** An **agent** uses the LLM as a *router*: given a question and a list of tools, the
LLM decides **which tool to call with what input**, runs it, and uses the result to answer. This
is the ReAct pattern (Reason + Act) at its simplest.

🖼️ **Diagram — the agent loop**
```
 question ─►[ LLM: which tool? ]─► pick tool + input ─►[ run tool ]─► observation ─►[ answer ]
```

🧠 **Mental model.** A chain has a *fixed* path; an agent *decides* the path at runtime. That
flexibility is powerful but less predictable — which is why the Agents lab (next!) goes deeper on
control and guardrails.


In [ ]:
def simple_agent(question, tools, llm):
    # In real LangChain the LLM emits a structured tool call. We simulate the routing:
    tool_names = ", ".join(t.name for t in tools)
    q = question.lower()
    # crude router (a real agent would let the LLM decide)
    if any(c.isdigit() for c in question) and any(op in question for op in "+-*/"):
        chosen, arg = "calculator", re.sub(r"[^0-9+\-*/(). ]", "", question)
    elif "how many words" in q or "word count" in q:
        chosen, arg = "word_count", question.split(":",1)[-1].strip()
    elif "uppercase" in q or "shout" in q:
        chosen, arg = "uppercase", question.split(":",1)[-1].strip()
    else:
        return llm.invoke(question)   # no tool needed
    tool = next(t for t in tools if t.name == chosen)
    observation = tool.run(arg)
    return f"[used {chosen}] {observation}"

print(simple_agent("what is 12 * (3 + 4)?", tools, llm))
print(simple_agent("uppercase this: make it loud", tools, llm))
print(simple_agent("how many words: the quick brown fox jumps", tools, llm))

⚡ **Pro tip.** Give each tool a **crisp description** — in a real agent, the LLM picks tools
purely from their names/descriptions. Vague descriptions cause wrong tool choices.

### ✏️ Your Turn 9.1
Add handling to `simple_agent` for your `reverse` tool (trigger on the word "reverse"), then test
`"reverse: hello world"`.

In [ ]:
# extend routing for the reverse tool and test it


✅ **Solution**
```python
# inside simple_agent, add before the else:
# elif "reverse" in q:
#     chosen, arg = "reverse", question.split(":",1)[-1].strip()
# (and include reverse_tool in the tools list)
```

---
## 🏆 Chapter 10 — Capstone: A Tool-Using Assistant Chain

Combine everything into an `Assistant` that: keeps **memory**, can **route to tools** when
needed, and otherwise answers via an **LLM chain**. Build it before revealing the solution.

In [ ]:
# Your Assistant here
class Assistant:
    def __init__(self, llm, tools):
        pass
    def ask(self, question):
        pass

# bot = Assistant(llm, tools)
# print(bot.ask("what is 6 * 7?"))
# print(bot.ask("what is the capital of France?"))


✅ **Capstone Solution**
```python
class Assistant:
    def __init__(self, llm, tools):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.memory = ConversationMemory()
    def _route(self, q):
        ql = q.lower()
        if any(c.isdigit() for c in q) and any(op in q for op in "+-*/"):
            return "calculator", re.sub(r"[^0-9+\-*/(). ]", "", q)
        if "uppercase" in ql or "shout" in ql:
            return "uppercase", q.split(":",1)[-1].strip()
        if "reverse" in ql:
            return "reverse", q.split(":",1)[-1].strip()
        return None, None
    def ask(self, question):
        name, arg = self._route(question)
        if name and name in self.tools:
            answer = f"[{name}] {self.tools[name].run(arg)}"
        else:
            prompt = f"History:\n{self.memory.windowed_load(2)}\nHuman: {question}\nAI:"
            answer = self.llm.invoke(prompt)
        self.memory.save(question, answer)
        return answer

# ensure reverse tool exists in the tools list
tools_all = tools + [Tool("reverse", "reverse a string", lambda s: s[::-1])]
bot = Assistant(llm, tools_all)
print(bot.ask("what is 6 * 7?"))
print(bot.ask("reverse: hello"))
print(bot.ask("what is the capital of France?"))
print("memory turns:", len(bot.memory.history))
```

🎉 **You understand LangChain from the inside out!** Runnables + the pipe operator, prompt
templates, chains (LCEL), output parsers, memory, retrieval chains, tools, and agents. The real
library adds polish and integrations, but every concept maps to what you just built.

---
### 📌 Real LangChain syntax (for reference)
```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_template("Translate '{text}' to {language}")
chain = prompt | llm | StrOutputParser()          # <-- the same | you learned
chain.invoke({"text": "hello", "language": "French"})
```

### 📌 Concept Quick-Reference
**Runnable:** anything with .invoke(input); compose with the | pipe (LCEL)
**PromptTemplate:** reusable prompt with {variables} (escape literals as {{ }})
**Chain:** prompt | llm | parser (single-responsibility stages)
**Output parsers:** text -> structured (JSON/dict), with error handling
**Memory:** store history; window or summarize to fit context
**Retrieval chain:** retriever -> format context -> llm (RAG as a chain)
**Tools:** name + description + func the model can call
**Agents:** LLM decides which tool to call at runtime (chain = fixed path, agent = decided path)
